In [ ]:
# Case 4: Prisoptimalisert drift med hensyn til vannverdi(Realistisk)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import milp, LinearConstraint, Bounds
from scipy.sparse import lil_matrix




# ============================================================
# 1) LAST FERDIG DATASETT
# ============================================================

df = pd.read_excel("Pris_2025.xlsx", engine="openpyxl")

df = df.rename(columns={
    "Dato, time": "Datetime",
    "Gjennomsnitt Pris (øre/kWh)": "Price_ore_kWh"
})

df["Datetime"] = pd.to_datetime(df["Datetime"])
df["Price_ore_kWh"] = pd.to_numeric(df["Price_ore_kWh"], errors="coerce")

df = df.dropna(subset=["Datetime", "Price_ore_kWh"]).copy()
df = df.sort_values("Datetime").reset_index(drop=True)

nT = len(df)
prices = df["Price_ore_kWh"].to_numpy()

# --------------------------------------------------
# FYLL MANGLENDE TIME I PRISDATA
# --------------------------------------------------

full_range = pd.date_range(
    start="2025-01-01 00:00:00",
    end="2025-12-31 23:00:00",
    freq="h"
)

df = df.set_index("Datetime").reindex(full_range)

df["Price_ore_kWh"] = df["Price_ore_kWh"].interpolate()

df = df.reset_index().rename(columns={"index": "Datetime"})

nT = len(df)
prices = df["Price_ore_kWh"].to_numpy()


# ============================================================
# 2) MAGASIN OG TILSIG
# ============================================================
# Antakelse:
# Vi har ikke ekte timesoppløst tilsig.
# Derfor lager vi et enkelt estimat:
# - enten bruker vi en konstant tilsig-verdi
# - eller dere kan senere bytte dette ut med bedre data
#
# Her bruker vi en enkel konstant timesverdi som start.
# Denne må justeres når dere får bedre grunnlag.
# ============================================================


# Eksempelverdier for magasinvolum (V) og tilsig
Vmax = 94.5e6     # m^3  (hentet fra nordkraft.no)
Vmin = 0.50e6
  
start_fyllingsgrad = 0.92
slutt_fyllingsgrad_min = 0.66
slutt_fyllingsgrad_max = 0.93

V0 = start_fyllingsgrad * Vmax      # startnivå: antatt 92 % fylt
V_slutt_min = slutt_fyllingsgrad_min * Vmax
V_slutt_max = slutt_fyllingsgrad_max * Vmax

# Estimert konstant tilsig per time

# --------------------------------------------------
# TILSIG BASERT PÅ NVE REGINE
# --------------------------------------------------

# NVE + konstant elv
annual_inflow_m3 = 158.998e6 # m³/år fra NVE REGINE

fordelingsfaktorer = np.array([
    0.3, 0.3, 0.4, 1.5, 2.5, 1.4,
    0.7, 0.7, 1.0, 1.4, 1.2, 0.6
])

dager_per_maaned = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])
timer_per_maaned = dager_per_maaned * 24

# Fordel årsvolumet på måneder
maanedsvolum = annual_inflow_m3 * fordelingsfaktorer / fordelingsfaktorer.sum()

# m³/time per måned
tilsig_per_maaned_h = maanedsvolum / timer_per_maaned

# Timeserie med tilsig
inflow_hourly = np.repeat(tilsig_per_maaned_h, timer_per_maaned)

# Tilpass lengde til datasettet
inflow_hourly = inflow_hourly[:len(df)]

# m³/s kun for kontroll
inflow_m3_per_s = inflow_hourly / 3600

print("Årlig tilsig:", inflow_hourly.sum() / 1e6, "millioner m³/år")
print("Gjennomsnittlig tilsig:", np.mean(inflow_m3_per_s), "m³/s")
print("Min tilsig:", np.min(inflow_m3_per_s), "m³/s")
print("Max tilsig:", np.max(inflow_m3_per_s), "m³/s")




# ============================================================
# 3) FYSISKE PARAMETERE
# ============================================================


rho = 1000        # kg/m^3
g = 9.81          # m/s^2
H_brutto = 655    # m
Q_max_1 = 10.7    # m^3/s
Q_max_2 = 16.05 
Q_max_3 = 21.4

# Driftstunnel 1
driftstunnel_1_L = 900      # m
driftstunnel_1_tverr = 9    # m^2

# Driftstunnel 2
driftstunnel_2_L = 1400
driftstunnel_2_tverr = 17

# Trykksjakt
trykksjakt_L = 550
trykksjakt_tverr = 5

# Overgang
overgang_L = 200
overgang_tverr = 15

# Pansring
pansring_L = 100
pansring_tverr = 2

# Utløp
utlop_L = 1200
utlop_tverr = 10.5


# --------------------------------------------------
# HYDRAULISK RADIUS
# R_h = D/4 fra R_h = A/P, der A = πD^2/4 og P = πD
# --------------------------------------------------
D_driftstunnel_1 = np.sqrt(4 * driftstunnel_1_tverr / np.pi)
R_h_driftstunnel_1 = D_driftstunnel_1 / 4

D_driftstunnel_2 = np.sqrt(4 * driftstunnel_2_tverr / np.pi)
R_h_driftstunnel_2 = D_driftstunnel_2 / 4

D_trykksjakt = np.sqrt(4 * trykksjakt_tverr / np.pi)
R_h_trykksjakt = D_trykksjakt / 4

D_overgang = np.sqrt(4 * overgang_tverr / np.pi)
R_h_overgang = D_overgang / 4

D_pansring = np.sqrt(4 * pansring_tverr / np.pi)
R_h_pansring = D_pansring / 4

D_utlop = np.sqrt(4 * utlop_tverr / np.pi)
R_h_utlop = D_utlop / 4

# --------------------------------------------------
# VANNHASTIGHETER
# --------------------------------------------------
# Hastigheter med Q_1 for alternativ 1 (~63 MW)
v1_driftstunnel_1 = Q_max_1 / driftstunnel_1_tverr
v1_driftstunnel_2 = Q_max_1 / driftstunnel_2_tverr
v1_trykksjakt = Q_max_1 / trykksjakt_tverr
v1_overgang = Q_max_1 / overgang_tverr
v1_pansring = Q_max_1 / pansring_tverr
v1_utlop = Q_max_1 / utlop_tverr

# Hastigheter med Q_2 for alternativ 2 (~94.5 MW)
v2_driftstunnel_1 = Q_max_2 / driftstunnel_1_tverr
v2_driftstunnel_2 = Q_max_2 / driftstunnel_2_tverr
v2_trykksjakt = Q_max_2 / trykksjakt_tverr
v2_overgang = Q_max_2 / overgang_tverr
v2_pansring = Q_max_2 /pansring_tverr
v2_utlop = Q_max_2 / utlop_tverr

# Hastigheter med Q_2 for alternativ 2 (~94.5 MW)
v3_driftstunnel_1 = Q_max_3 / driftstunnel_1_tverr
v3_driftstunnel_2 = Q_max_3 / driftstunnel_2_tverr
v3_trykksjakt = Q_max_3 / trykksjakt_tverr
v3_overgang = Q_max_3 / overgang_tverr
v3_pansring = Q_max_3 /pansring_tverr
v3_utlop = Q_max_3 / utlop_tverr


# --------------------------------------------------
# MANNING-STRICKLER RUHETSKOEFFISIENTER
# --------------------------------------------------
M_driftstunnel_1 = 35
M_driftstunnel_2 = 35
M_trykksjakt = 65
M_overgang = 35
M_pansring = 100
M_utlop = 35

# --------------------------------------------------
# FALLTAP UT FRA STRICKLER-FORMELEN
# h_f = L * v^2 / (M^2 * R_h^(4/3))
# --------------------------------------------------
# Falltap med Q_1
h_f_driftstunnel_1_Q1 = driftstunnel_1_L * v1_driftstunnel_1**2 / (M_driftstunnel_1**2 * R_h_driftstunnel_1**(4/3))
h_f_driftstunnel_2_Q1 = driftstunnel_2_L * v1_driftstunnel_2**2 / (M_driftstunnel_2**2 * R_h_driftstunnel_2**(4/3))
h_f_trykksjakt_Q1     = trykksjakt_L     * v1_trykksjakt**2     / (M_trykksjakt**2     * R_h_trykksjakt**(4/3))
h_f_overgang_Q1       = overgang_L       * v1_overgang**2       / (M_overgang**2       * R_h_overgang**(4/3))
h_f_pansring_Q1       = pansring_L       * v1_pansring**2       / (M_pansring**2       * R_h_pansring**(4/3))
h_f_utlop_Q1          = utlop_L          * v1_utlop**2          / (M_utlop**2          * R_h_utlop**(4/3))

# Falltap med Q_2
h_f_driftstunnel_1_Q2 = driftstunnel_1_L * v2_driftstunnel_1**2  / (M_driftstunnel_1**2 * R_h_driftstunnel_1**(4/3))
h_f_driftstunnel_2_Q2 = driftstunnel_2_L * v2_driftstunnel_2**2  / (M_driftstunnel_2**2 * R_h_driftstunnel_2**(4/3))
h_f_trykksjakt_Q2     = trykksjakt_L     * v2_trykksjakt**2      / (M_trykksjakt**2     * R_h_trykksjakt**(4/3))
h_f_overgang_Q2       = overgang_L       * v2_overgang**2        / (M_overgang**2       * R_h_overgang**(4/3))
h_f_pansring_Q2       = pansring_L       * v2_pansring**2        / (M_pansring**2       * R_h_pansring**(4/3))
h_f_utlop_Q2          = utlop_L          * v2_utlop**2           / (M_utlop**2          * R_h_utlop**(4/3))

# Falltap med Q_2
h_f_driftstunnel_1_Q3 = driftstunnel_1_L * v3_driftstunnel_1**2  / (M_driftstunnel_1**2 * R_h_driftstunnel_1**(4/3))
h_f_driftstunnel_2_Q3 = driftstunnel_2_L * v3_driftstunnel_2**2  / (M_driftstunnel_2**2 * R_h_driftstunnel_2**(4/3))
h_f_trykksjakt_Q3     = trykksjakt_L     * v3_trykksjakt**2      / (M_trykksjakt**2     * R_h_trykksjakt**(4/3))
h_f_overgang_Q3       = overgang_L       * v3_overgang**2        / (M_overgang**2       * R_h_overgang**(4/3))
h_f_pansring_Q3       = pansring_L       * v3_pansring**2        / (M_pansring**2       * R_h_pansring**(4/3))
h_f_utlop_Q3          = utlop_L          * v3_utlop**2           / (M_utlop**2          * R_h_utlop**(4/3))

# Totalt falltap
h_f_total_Q1 = (
    h_f_driftstunnel_1_Q1 + h_f_driftstunnel_2_Q1 + h_f_trykksjakt_Q1 +
    h_f_overgang_Q1 + h_f_pansring_Q1 + h_f_utlop_Q1
)

h_f_total_Q2 = (
    h_f_driftstunnel_1_Q2 + h_f_driftstunnel_2_Q2 + h_f_trykksjakt_Q2 +
    h_f_overgang_Q2 + h_f_pansring_Q2 + h_f_utlop_Q2
)
h_f_total_Q3 = (
    h_f_driftstunnel_1_Q3 + h_f_driftstunnel_2_Q3 + h_f_trykksjakt_Q3 +
    h_f_overgang_Q3 + h_f_pansring_Q3 + h_f_utlop_Q3
)

# --------------------------------------------------
# NETTO FALLHØYDE OG VIRKNINGSGRAD
# --------------------------------------------------

H_netto_1 = H_brutto - h_f_total_Q1
H_netto_2 = H_brutto - h_f_total_Q2
H_netto_3 = H_brutto - h_f_total_Q3

eta_vannvei_1 = H_netto_1 / H_brutto
eta_vannvei_2 = H_netto_2 / H_brutto
eta_vannvei_3 = H_netto_3 / H_brutto

eta_pelton = 0.92
eta_generator = 0.99
eta_transformator = 0.99

eta_total_1 = eta_vannvei_1 * eta_pelton * eta_generator * eta_transformator
eta_total_2 = eta_vannvei_2 * eta_pelton * eta_generator * eta_transformator



#-----------------------------------------------------
#-------------------CASE------------------------------
#-----------------------------------------------------



def run_case4_global(scenario_name, Q_max, H_netto_scenario, P_min):

    # Diskrete vannføringsnivåer
    
    max_delta_Q = 21.4
    #Q_levels = np.arange(0, Q_max + 0.01, 1)
    #Q_levels = np.arange(0, Q_max + 0.01, delta_Q_step)
    Q_levels = np.linspace(0, Q_max, 10)

    def eta_turbin_local(Q):
        if Q <= 0:
            return 0.0

        q_rel = Q / Q_max

        if q_rel < 0.25:
            return 0.0
        elif q_rel < 0.50:
            return 0.65 + (q_rel - 0.30) * (0.88 - 0.65) / 0.20
        elif q_rel < 0.90:
            return 0.85 + 0.07 * (1 - ((q_rel - 0.70) / 0.20)**2)
        elif q_rel <= 1.00:
            return 0.88
        else:
            return 0.0

    def power_MW_local(Q):
        eta = eta_turbin_local(Q) * eta_generator * eta_transformator
        P = rho * g * Q * H_netto_scenario * eta / 1e6
        return P, eta

    P_levels = np.array([power_MW_local(Q)[0] for Q in Q_levels])
    eta_levels = np.array([power_MW_local(Q)[1] for Q in Q_levels])

    # Ugyldige nivåer: drift under P_min
    valid = np.ones(len(Q_levels), dtype=bool)
    valid[(Q_levels > 0) & (P_levels < P_min)] = False

    K = len(Q_levels)
    T = nT

    # Variabler:
    # x[t,k] = 1 hvis vannføring nivå k velges i time t
    # V[t] = magasinvolum ved slutten av time t
    # y[t] = 1 hvis oppstart i time t

    n_x = T * K
    n_V = T
    n_y = T
    n_vars = n_x + n_V + n_y

    def idx_x(t, k):
        return t * K + k

    def idx_V(t):
        return n_x + t

    def idx_y(t):
        return n_x + n_V + t

    c = np.zeros(n_vars)

    startup_cost_ore = 750000

    # scipy milp minimerer, derfor bruker vi negativ profitt
    for t in range(T):
        for k in range(K):
            revenue = prices[t] * 1000 * P_levels[k]
            c[idx_x(t, k)] = -revenue

        c[idx_y(t)] = startup_cost_ore

    constraints = []
    lb = []
    ub = []

    # 1) Akkurat ett Q-nivå per time
    for t in range(T):
        row = lil_matrix((1, n_vars))
        for k in range(K):
            row[0, idx_x(t, k)] = 1
        constraints.append(row)
        lb.append(1)
        ub.append(1)

    # 0) Startbetingelse: starter fra 0 (ramp inn i første time)
    row = lil_matrix((1, n_vars))
    for k in range(K):
        row[0, idx_x(0, k)] = Q_levels[k]

    constraints.append(row)
    lb.append(-np.inf)
    ub.append(max_delta_Q )




    # 2) Magasinbalanse
    for t in range(T):
        row = lil_matrix((1, n_vars))

        row[0, idx_V(t)] = 1

        if t > 0:
            row[0, idx_V(t - 1)] = -1

        for k in range(K):
            row[0, idx_x(t, k)] = Q_levels[k] * 3600

        rhs = (V0 + inflow_hourly[t]) if t == 0 else inflow_hourly[t]

        constraints.append(row)
        lb.append(rhs)
        ub.append(rhs)

    # 3) Sluttkrav magasin
    row = lil_matrix((1, n_vars))
    row[0, idx_V(T - 1)] = 1
    constraints.append(row)
    lb.append(V_slutt_min)
    ub.append(V_slutt_max)

    # 4) Ugyldige Q-nivåer forbys
    for t in range(T):
        for k in range(K):
            if not valid[k]:
                row = lil_matrix((1, n_vars))
                row[0, idx_x(t, k)] = 1
                constraints.append(row)
                lb.append(0)
                ub.append(0)

    # 5) Oppstart: y[t] >= on[t] - on[t-1]
    # on[t] = sum av alle x[t,k] der Q > 0
    for t in range(T):
        row = lil_matrix((1, n_vars))

        for k in range(K):
            if Q_levels[k] > 0:
                row[0, idx_x(t, k)] = 1

        if t > 0:
            for k in range(K):
                if Q_levels[k] > 0:
                    row[0, idx_x(t - 1, k)] -= 1

        row[0, idx_y(t)] = -1

        constraints.append(row)
        lb.append(-np.inf)
        ub.append(0)


    
    # 6) Ramp-begrensning
    for t in range(1, T):

        row = lil_matrix((1, n_vars))
        for k in range(K):
            row[0, idx_x(t, k)] += Q_levels[k]
            row[0, idx_x(t - 1, k)] -= Q_levels[k]

        constraints.append(row)
        lb.append(-np.inf)
        ub.append(max_delta_Q)

        row = lil_matrix((1, n_vars))
        for k in range(K):
            row[0, idx_x(t - 1, k)] += Q_levels[k]
            row[0, idx_x(t, k)] -= Q_levels[k]

        constraints.append(row)
        lb.append(-np.inf)
        ub.append(max_delta_Q)

    A = lil_matrix((len(constraints), n_vars))
    for i, row in enumerate(constraints):
        A[i, :] = row

    linear_constraint = LinearConstraint(A.tocsr(), np.array(lb), np.array(ub))

    lower = np.zeros(n_vars)
    upper = np.ones(n_vars)

    # Magasinvariabler er kontinuerlige
    for t in range(T):
        lower[idx_V(t)] = Vmin
        upper[idx_V(t)] = Vmax

    integrality = np.ones(n_vars)
    for t in range(T):
        integrality[idx_V(t)] = 0

    result = milp(
        c=c,
        integrality=integrality,
        bounds=Bounds(lower, upper),
        constraints=linear_constraint,
        options={
            "time_limit": 300,
            "mip_rel_gap": 0.03
        }
    )

    if not result.success:
        print("Optimalisering fant ikke perfekt løsning:")
        print(result.message)

    z = result.x

    Q_out = np.zeros(T)
    P_out = np.zeros(T)
    eta_out = np.zeros(T)
    V_out = np.zeros(T)

    for t in range(T):
        chosen_k = np.argmax([z[idx_x(t, k)] for k in range(K)])
        Q_out[t] = Q_levels[chosen_k]
        P_out[t] = P_levels[chosen_k]
        eta_out[t] = eta_levels[chosen_k]
        V_out[t] = z[idx_V(t)]

    
    
    revenue_ore = prices[:T] * 1000 * P_out

    results = pd.DataFrame({
        "Scenario": scenario_name,
        "Datetime": df["Datetime"].iloc[:T].to_numpy(),
        "Price_ore_kWh": prices[:T],
        "Q_opt_m3s": Q_out,
        "P_opt_MW": P_out,
        "eta_opt": eta_out,
        "H_netto_m": H_netto_scenario,
        "Objective_ore_per_h": revenue_ore,
        "Magasin_m3": V_out
    })

    return results
    






results_60MW = run_case4_global(
    scenario_name="60 MW global",
    Q_max=Q_max_1,
    H_netto_scenario=H_netto_1,
    P_min=0
)

results_90MW = run_case4_global(
    scenario_name="90 MW global",
    Q_max=Q_max_2,
    H_netto_scenario=H_netto_2,
    P_min=0
)
results_120MW = run_case4_global(
    scenario_name="120 MW global",
    Q_max=Q_max_3,
    H_netto_scenario=H_netto_3,
    P_min=0
)






# ============================================================
# 9) RESULTATER OG NØKKELTALL FOR BEGGE SCENARIOER
# ============================================================

all_results = pd.concat([results_60MW, results_90MW, results_120MW], ignore_index=True)

def print_key_numbers(results, navn):
    total_energy_MWh = results["P_opt_MW"].sum()
    total_profit_NOK = results["Objective_ore_per_h"].sum() / 100
    avg_eta = results.loc[results["P_opt_MW"] > 0, "eta_opt"].mean()
    drift_hours = (results["P_opt_MW"] > 0).sum()

    starts = (
        (results["Q_opt_m3s"] > 0) &
        (results["Q_opt_m3s"].shift(1, fill_value=0) == 0)
    ).sum()

    print(f"\n=== NØKKELTALL CASE 4 – {navn} ===")
    print(f"Total energi: {total_energy_MWh:.1f} MWh")
    print(f"Total objektverdi: {total_profit_NOK:,.0f} kr")
    print(f"Gjennomsnittlig virkningsgrad i drift: {avg_eta*100:.2f} %")
    print(f"Driftstimer: {drift_hours}")
    print(f"Antall oppstarter: {starts}")
    print(f"Maks effekt: {results['P_opt_MW'].max():.2f} MW")
    print(f"Maks vannføring: {results['Q_opt_m3s'].max():.2f} m³/s")


print_key_numbers(results_60MW, "60 MW")
print_key_numbers(results_90MW, "90 MW")
print_key_numbers(results_120MW, "120 MW")


summary = all_results.groupby("Scenario").agg(
    Total_energi_MWh=("P_opt_MW", "sum"),
    Total_objektverdi_kr=("Objective_ore_per_h", lambda x: x.sum() / 100),
    Driftstimer=("P_opt_MW", lambda x: (x > 0).sum()),
    Maks_effekt_MW=("P_opt_MW", "max"),
    Snitt_effekt_MW=("P_opt_MW", "mean"),
    Snitt_eta=("eta_opt", lambda x: x[x > 0].mean()),
    Maks_Q_m3s=("Q_opt_m3s", "max")
)

print("\n=== SAMMENLIGNING AV SCENARIOER ===")
print(summary)



# ============================================================
# 10) PLOTS CASE 4 – BEGGE SCENARIOER
# ============================================================

# 1. Produksjon
plt.figure(figsize=(14, 5))
plt.plot(results_60MW["Datetime"], results_60MW["P_opt_MW"], label="60 MW")
plt.plot(results_90MW["Datetime"], results_90MW["P_opt_MW"], label="90 MW")
plt.plot(results_120MW["Datetime"], results_120MW["P_opt_MW"], label="120 MW")
plt.title("Case 4 – optimal produksjon for 60 MW og 90 MW")
plt.ylabel("Effekt (MW)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


# 2. Vannføring
plt.figure(figsize=(14, 5))
plt.plot(results_60MW["Datetime"], results_60MW["Q_opt_m3s"], label="60 MW")
plt.plot(results_90MW["Datetime"], results_90MW["Q_opt_m3s"], label="90 MW")
plt.plot(results_120MW["Datetime"], results_120MW["Q_opt_m3s"], label="120 MW")
plt.title("Case 4 – optimal vannføring for 60 MW og 90 MW")
plt.ylabel("Q (m³/s)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


# 3. Magasin
plt.figure(figsize=(14, 5))
plt.plot(results_60MW["Datetime"], results_60MW["Magasin_m3"] / 1e6, label="60 MW")
plt.plot(results_90MW["Datetime"], results_90MW["Magasin_m3"] / 1e6, label="90 MW")
plt.plot(results_120MW["Datetime"], results_120MW["Magasin_m3"] / 1e6, label="120 MW")
plt.axhline(Vmin / 1e6, color="red", linestyle="--", label="Vmin")
plt.title("Case 4 – magasinutvikling for 60 MW og 90 MW")
plt.ylabel("Magasinvolum (millioner m³)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


# 4. Ukemidlet pris og produksjon
results_60_plot = results_60MW.set_index("Datetime")
results_90_plot = results_90MW.set_index("Datetime")
results_120_plot = results_120MW.set_index("Datetime")

weekly_60 = results_60_plot.resample("W").mean(numeric_only=True)
weekly_90 = results_90_plot.resample("W").mean(numeric_only=True)
weekly_120 = results_120_plot.resample("W").mean(numeric_only=True)

plt.figure(figsize=(14, 5))

ax1 = plt.gca()
ax1.plot(weekly_60.index, weekly_60["Price_ore_kWh"], label="Pris", linewidth=2, color="green")
ax1.set_ylabel("Pris (øre/kWh)")
ax1.set_title("Case 4 – ukemidlet pris og produksjon")
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(weekly_60.index, weekly_60["P_opt_MW"], label="Produksjon 60 MW", linewidth=2)
ax2.plot(weekly_90.index, weekly_90["P_opt_MW"], label="Produksjon 90 MW", linewidth=2)
ax2.plot(weekly_120.index, weekly_120["P_opt_MW"], label="Produksjon 120 MW", linewidth=2)
ax2.set_ylabel("Produksjon (MW)")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2)

plt.show()





Årlig tilsig: 158.998 millioner m³/år
Gjennomsnittlig tilsig: 5.041793505834602 m³/s
Min tilsig: 1.4840763142174433 m³/s
Max tilsig: 12.367302618478694 m³/s
